# 01 - Data Ingestion & Cleaning

This notebook loads all Aadhaar datasets, performs cleaning, and saves processed files.

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
from pathlib import Path

# Set random seed for reproducibility
np.random.seed(42)

# Project paths
BASE_DIR = Path('..')
DATA_DIR = BASE_DIR
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print('Paths configured successfully')

## 1.1 Load Raw Data

In [ ]:
def load_dataset(pattern, name):
    """Load all CSVs matching pattern and concatenate."""
    files = glob.glob(str(DATA_DIR / pattern))
    print(f"Loading {name}: {len(files)} files found")
    
    dfs = []
    for f in sorted(files):
        df = pd.read_csv(f)
        print(f"  - {os.path.basename(f)}: {len(df):,} rows")
        dfs.append(df)
    
    combined = pd.concat(dfs, ignore_index=True)
    print(f"  Total: {len(combined):,} rows\n")
    return combined

# Load all three datasets
df_enrol = load_dataset('api_data_aadhar_enrolment/*.csv', 'Enrolment')
df_bio = load_dataset('api_data_aadhar_biometric/*.csv', 'Biometric')
df_demo = load_dataset('api_data_aadhar_demographic/*.csv', 'Demographic')

## 1.2 Initial Data Inspection

In [ ]:
def inspect_dataset(df, name):
    """Print basic info about a dataset."""
    print(f"=== {name} ===")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print(f"\nData types:\n{df.dtypes}")
    print(f"\nMissing values:\n{df.isnull().sum()}")
    print(f"\nSample rows:")
    display(df.head())
    print("\n" + "="*50 + "\n")

inspect_dataset(df_enrol, 'Enrolment')
inspect_dataset(df_bio, 'Biometric')
inspect_dataset(df_demo, 'Demographic')

## 1.3 Data Cleaning

In [ ]:
def clean_dataset(df, name):
    """Apply standard cleaning operations."""
    print(f"Cleaning {name}...")
    original_rows = len(df)
    
    # 1. Strip whitespace from column names
    df.columns = df.columns.str.strip()
    
    # 2. Strip whitespace from string columns
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].str.strip()
    
    # 3. Parse date column
    df['date'] = pd.to_datetime(df['date'], format='%d-%m-%Y', errors='coerce')
    
    # 4. Standardize state names (title case)
    df['state'] = df['state'].str.title()
    
    # 5. Handle negative values in numeric columns (set to 0)
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        negatives = (df[col] < 0).sum()
        if negatives > 0:
            print(f"  - Found {negatives} negative values in '{col}', setting to 0")
            df[col] = df[col].clip(lower=0)
    
    # 6. Remove duplicate rows
    duplicates = df.duplicated().sum()
    if duplicates > 0:
        print(f"  - Removing {duplicates} duplicate rows")
        df = df.drop_duplicates()
    
    # 7. Drop rows with missing critical columns
    critical_cols = ['date', 'state', 'district', 'pincode']
    missing_critical = df[critical_cols].isnull().any(axis=1).sum()
    if missing_critical > 0:
        print(f"  - Dropping {missing_critical} rows with missing critical data")
        df = df.dropna(subset=critical_cols)
    
    print(f"  Cleaned: {original_rows:,} -> {len(df):,} rows\n")
    return df

df_enrol = clean_dataset(df_enrol, 'Enrolment')
df_bio = clean_dataset(df_bio, 'Biometric')
df_demo = clean_dataset(df_demo, 'Demographic')

## 1.4 Feature Engineering

In [ ]:
# Enrolment: Add total and fractions
df_enrol['total_enrol'] = df_enrol['age_0_5'] + df_enrol['age_5_17'] + df_enrol['age_18_greater']
df_enrol['child_fraction'] = df_enrol['age_0_5'] / df_enrol['total_enrol'].replace(0, np.nan)
df_enrol['youth_fraction'] = df_enrol['age_5_17'] / df_enrol['total_enrol'].replace(0, np.nan)
df_enrol['adult_fraction'] = df_enrol['age_18_greater'] / df_enrol['total_enrol'].replace(0, np.nan)

# Biometric: Add total
df_bio['total_bio'] = df_bio['bio_age_5_17'] + df_bio['bio_age_17_']

# Demographic: Add total
df_demo['total_demo'] = df_demo['demo_age_5_17'] + df_demo['demo_age_17_']

# Add temporal features to all datasets
for df in [df_enrol, df_bio, df_demo]:
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['week'] = df['date'].dt.isocalendar().week
    df['day_of_week'] = df['date'].dt.dayofweek
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

print("Feature engineering complete.")
print(f"Enrolment columns: {list(df_enrol.columns)}")
print(f"Biometric columns: {list(df_bio.columns)}")
print(f"Demographic columns: {list(df_demo.columns)}")

## 1.5 Data Quality Summary

In [ ]:
def summarize_dataset(df, name):
    """Print summary statistics."""
    print(f"=== {name} Summary ===")
    print(f"Records: {len(df):,}")
    print(f"Date range: {df['date'].min()} to {df['date'].max()}")
    print(f"States: {df['state'].nunique()}")
    print(f"Districts: {df['district'].nunique()}")
    print(f"Pincodes: {df['pincode'].nunique()}")
    print()

summarize_dataset(df_enrol, 'Enrolment')
summarize_dataset(df_bio, 'Biometric')
summarize_dataset(df_demo, 'Demographic')

## 1.6 Save Processed Data

In [ ]:
# Save as Parquet for efficient storage and loading
df_enrol.to_parquet(PROCESSED_DIR / 'enrolment_cleaned.parquet', index=False)
df_bio.to_parquet(PROCESSED_DIR / 'biometric_cleaned.parquet', index=False)
df_demo.to_parquet(PROCESSED_DIR / 'demographic_cleaned.parquet', index=False)

print("Processed data saved to:")
for f in PROCESSED_DIR.glob('*.parquet'):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  - {f.name}: {size_mb:.2f} MB")

In [ ]:
print("\n✅ Data ingestion and cleaning complete!")
print("Proceed to notebook 02 for EDA and visualizations.")